# Brain Graph Super-Resolution using GCN with Bilinear Edge Decoder (BrainGCN-SR)

Predicts high-resolution (268x268) brain graphs from low-resolution (160x160) brain graphs using a 3-layer GCN with multi-scale features, learned upscaling, and a bilinear edge decoder. Permutation equivariant throughout.

In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
from scipy.spatial.distance import jensenshannon
import networkx as nx
import matplotlib.pyplot as plt


def vectorize(matrix):
    """Convert symmetric matrix to vector (column-major upper triangle, no diagonal)."""
    n = matrix.shape[0]
    elements = []
    for col in range(n):
        for row in range(col):
            elements.append(matrix[row, col])
    return np.array(elements)


def anti_vectorize(vector, n):
    """Reconstruct symmetric matrix from vector (column-major upper triangle, no diagonal)."""
    matrix = np.zeros((n, n))
    idx = 0
    for col in range(n):
        for row in range(col):
            matrix[row, col] = vector[idx]
            matrix[col, row] = vector[idx]
            idx += 1
    return matrix

## Reproducibility & Device Setup

In [ ]:
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available. Using GPU.")
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")

## Model Definition

In [ ]:
class GCNLayer(nn.Module):
    """Single GCN layer: X' = sigma(D^{-1/2} A D^{-1/2} X W)"""

    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj_norm):
        support = x @ self.weight
        out = adj_norm @ support
        return out + self.bias


HR_VEC_SIZE = 35778  # upper triangular of 268x268 = 268*267/2


class BrainGCN_SR(nn.Module):
    """
    GCN encoder with multi-scale features + learned upscaling + bilinear edge decoder.

    Pipeline (permutation equivariant throughout):
        1. Normalize LR adjacency with self-loops
        2. Use LR adjacency rows as initial node features
        3. N-layer GCN encoder, concatenate all layer outputs (multi-scale)
        4. Project multi-scale features to edge embedding space
        5. Learned upscaling matrix U maps 160 LR -> 268 HR node embeddings
        6. Bilinear edge decoder: A[i,j] = sigma(z_i^T W z_j)
    """

    def __init__(self, lr_dim=160, hr_dim=268, hidden_dims=None,
                 edge_dim=128, dropout=0.1):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [256, 256, 256]
        self.lr_dim = lr_dim
        self.hr_dim = hr_dim
        self.hidden_dims = hidden_dims
        self.num_layers = len(hidden_dims)

        # GCN encoder layers (dynamic)
        self.gcn_layers = nn.ModuleList()
        in_dim = lr_dim
        for out_dim in hidden_dims:
            self.gcn_layers.append(GCNLayer(in_dim, out_dim))
            in_dim = out_dim
        self.dropout = nn.Dropout(dropout)

        # Multi-scale feature dimension (concat ALL layer outputs)
        multi_scale_dim = sum(hidden_dims)

        # 2-layer MLP projection: adds per-node nonlinearity (equivariant)
        self.proj = nn.Sequential(
            nn.Linear(multi_scale_dim, edge_dim),
            nn.ReLU(),
            nn.Linear(edge_dim, edge_dim),
        )

        # Learned upscaling matrix: maps LR (160) node space -> HR (268) node space
        self.U = nn.Parameter(torch.empty(hr_dim, lr_dim))
        nn.init.xavier_uniform_(self.U)

        # Bilinear edge decoder weight matrix
        self.W_edge = nn.Parameter(torch.empty(edge_dim, edge_dim))
        nn.init.xavier_uniform_(self.W_edge)

        # Precompute vectorization indices (column-major upper triangle, matching MatrixVectorizer)
        rows, cols = [], []
        for col in range(hr_dim):
            for row in range(col):
                rows.append(row)
                cols.append(col)
        self.register_buffer("vec_rows", torch.LongTensor(rows))
        self.register_buffer("vec_cols", torch.LongTensor(cols))

    def normalize_adj(self, adj):
        """Symmetric normalization: D^{-1/2} (A + I) D^{-1/2}"""
        adj = adj + torch.eye(adj.size(0), device=adj.device)
        degree = adj.sum(dim=1)
        d_inv_sqrt = degree.pow(-0.5)
        d_inv_sqrt[d_inv_sqrt == float("inf")] = 0.0
        return (adj * d_inv_sqrt.unsqueeze(1)) * d_inv_sqrt.unsqueeze(0)

    def forward(self, adj_lr):
        adj_norm = self.normalize_adj(adj_lr)
        x = adj_lr  # (160, 160) -- node features = connectivity profile

        # N-layer GCN with multi-scale feature collection + residual connections
        layer_outputs = []
        h = x
        for i, gcn in enumerate(self.gcn_layers):
            h_new = gcn(h, adj_norm)
            # ReLU + dropout on all layers except the last
            if i < self.num_layers - 1:
                h_new = self.dropout(F.relu(h_new))
            # Residual: add most recent previous layer with matching dimension
            for j in range(len(layer_outputs) - 1, -1, -1):
                if layer_outputs[j].shape[1] == h_new.shape[1]:
                    h_new = h_new + layer_outputs[j]
                    break
            layer_outputs.append(h_new)
            h = h_new

        # Concatenate multi-scale features (all layer outputs)
        z_lr = torch.cat(layer_outputs, dim=1)  # (160, sum(hidden_dims))

        # 2-layer MLP projection to edge embedding space
        z_lr = self.proj(z_lr)  # (160, edge_dim)

        # Upscale to HR node space
        z_hr = self.U @ z_lr  # (268, edge_dim)

        # Bilinear edge decoder
        a_raw = z_hr @ self.W_edge @ z_hr.t()  # (268, 268)
        a_sym = (a_raw + a_raw.t()) / 2  # ensure symmetry
        a_pred = torch.sigmoid(a_sym)  # (268, 268), values in [0, 1]

        # Vectorize upper triangle (column-major order, matching MatrixVectorizer)
        hr_vec = a_pred[self.vec_rows, self.vec_cols]  # (35778,)

        return hr_vec

## Data Loading

In [ ]:
def load_data(lr_path, hr_path):
    """Load LR as matrices (for GCN input) and HR as vectors (for loss target)."""
    lr_vecs = pd.read_csv(lr_path).values
    hr_vecs = pd.read_csv(hr_path).values

    # Anti-vectorize LR into matrices for GCN input
    lr_matrices = np.array([anti_vectorize(lr_vecs[i], 160)
                            for i in range(lr_vecs.shape[0])])
    return lr_matrices, hr_vecs


def load_test_data(test_lr_path):
    """Load test LR as matrices."""
    lr_vecs = pd.read_csv(test_lr_path).values
    lr_matrices = np.array([anti_vectorize(lr_vecs[i], 160)
                            for i in range(lr_vecs.shape[0])])
    return lr_matrices

## Training

In [ ]:
def train_model(model, train_lr_matrices, train_hr_vecs, val_lr_matrices=None,
                val_hr_vecs=None, epochs=200, lr=0.001, device="cpu", patience=50,
                noise_std=0.005):
    """
    Train the model with early stopping. LR input is matrices (for GCN), HR target is vectors.
    Adds Gaussian noise to LR inputs during training for data augmentation.
    """
    model.to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    # Early stopping state
    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        perm = np.random.permutation(len(train_lr_matrices))
        for idx in perm:
            adj_lr = torch.FloatTensor(train_lr_matrices[idx]).to(device)
            # Data augmentation: add small Gaussian noise to LR input
            if noise_std > 0:
                noise = torch.randn_like(adj_lr) * noise_std
                adj_lr = (adj_lr + noise).clamp(min=0.0)
                # Re-symmetrize after noise
                adj_lr = (adj_lr + adj_lr.t()) / 2
            hr_vec = torch.FloatTensor(train_hr_vecs[idx]).to(device)

            optimizer.zero_grad()
            pred_vec = model(adj_lr)
            loss = F.l1_loss(pred_vec, hr_vec)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()

        scheduler.step()

        avg_train_loss = epoch_loss / len(train_lr_matrices)

        # Validation
        avg_val_loss = None
        if val_lr_matrices is not None and val_hr_vecs is not None:
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for idx in range(len(val_lr_matrices)):
                    adj_lr = torch.FloatTensor(val_lr_matrices[idx]).to(device)
                    hr_vec = torch.FloatTensor(val_hr_vecs[idx]).to(device)
                    pred_vec = model(adj_lr)
                    val_loss += F.l1_loss(pred_vec, hr_vec).item()
            avg_val_loss = val_loss / len(val_lr_matrices)

            # Early stopping check
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch+1} (best val MAE: {best_val_loss:.6f})")
                break

        if (epoch + 1) % 20 == 0:
            msg = f"Epoch {epoch+1}/{epochs} | Train MAE: {avg_train_loss:.6f}"
            if avg_val_loss is not None:
                msg += f" | Val MAE: {avg_val_loss:.6f}"
            print(msg)

    # Restore best model weights
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"  Restored best model (val MAE: {best_val_loss:.6f})")

## Prediction & Evaluation

In [ ]:
def predict(model, lr_matrices, device="cpu"):
    """Generate HR vector predictions for a set of LR matrices."""
    model.eval()
    pred_vecs = []
    with torch.no_grad():
        for i in range(len(lr_matrices)):
            adj_lr = torch.FloatTensor(lr_matrices[i]).to(device)
            pred_vec = model(adj_lr)
            pred_vec = pred_vec.clamp(min=0.0)
            pred_vecs.append(pred_vec.cpu().numpy())
    return np.array(pred_vecs)


def compute_evaluation_metrics(pred_vecs, gt_vecs):
    """
    Compute all 8 evaluation metrics following the official evaluation code.
    """
    num_samples = len(pred_vecs)
    metrics = {}

    mae_bc, mae_ec, mae_pc, mae_cc, mae_dc = [], [], [], [], []

    for i in range(num_samples):
        print(f"    Sample {i+1}/{num_samples}...", end="\r")

        pred_mat = anti_vectorize(pred_vecs[i], 268)
        gt_mat = anti_vectorize(gt_vecs[i], 268)

        pred_graph = nx.from_numpy_array(pred_mat, edge_attr="weight")
        gt_graph = nx.from_numpy_array(gt_mat, edge_attr="weight")

        pred_bc = nx.betweenness_centrality(pred_graph, weight="weight", k=50)
        pred_ec = nx.eigenvector_centrality(pred_graph, weight="weight", max_iter=1000)
        pred_pc = nx.pagerank(pred_graph, weight="weight")

        gt_bc = nx.betweenness_centrality(gt_graph, weight="weight", k=50)
        gt_ec = nx.eigenvector_centrality(gt_graph, weight="weight", max_iter=1000)
        gt_pc = nx.pagerank(gt_graph, weight="weight")

        mae_bc.append(mean_absolute_error(list(pred_bc.values()), list(gt_bc.values())))
        mae_ec.append(mean_absolute_error(list(pred_ec.values()), list(gt_ec.values())))
        mae_pc.append(mean_absolute_error(list(pred_pc.values()), list(gt_pc.values())))

        pred_cc = nx.closeness_centrality(pred_graph)
        gt_cc = nx.closeness_centrality(gt_graph)
        mae_cc.append(mean_absolute_error(list(pred_cc.values()), list(gt_cc.values())))

        pred_dc = nx.degree_centrality(pred_graph)
        gt_dc = nx.degree_centrality(gt_graph)
        mae_dc.append(mean_absolute_error(list(pred_dc.values()), list(gt_dc.values())))

    pred_1d = np.concatenate([pred_vecs[i] for i in range(num_samples)])
    gt_1d = np.concatenate([gt_vecs[i] for i in range(num_samples)])

    metrics["MAE"] = mean_absolute_error(pred_1d, gt_1d)
    metrics["PCC"] = pearsonr(pred_1d, gt_1d)[0]
    metrics["JSD"] = jensenshannon(pred_1d, gt_1d)
    metrics["MAE(PC)"] = np.mean(mae_pc)
    metrics["MAE(EC)"] = np.mean(mae_ec)
    metrics["MAE(BC)"] = np.mean(mae_bc)
    metrics["MAE(CC)"] = np.mean(mae_cc)
    metrics["MAE(DC)"] = np.mean(mae_dc)

    return metrics

## Plotting

In [ ]:
def plot_fold_results(all_fold_metrics, output_dir="outputs"):
    """Generate bar plots for each fold and averaged across folds."""
    os.makedirs(output_dir, exist_ok=True)

    metric_names = list(all_fold_metrics[0].keys())
    n_folds = len(all_fold_metrics)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    colors = ["#00CED1", "#FF6347", "#FFD700", "#32CD32", "#8A2BE2", "#FF69B4", "#FF8C00", "#1E90FF"]

    for fold_idx in range(n_folds):
        ax = axes[fold_idx]
        values = [all_fold_metrics[fold_idx][m] for m in metric_names]
        ax.bar(range(len(metric_names)), values, color=colors[:len(metric_names)])
        ax.set_title(f"Fold {fold_idx + 1}", fontsize=13)
        ax.set_xticks(range(len(metric_names)))
        ax.set_xticklabels(metric_names, rotation=45, ha="right", fontsize=9)
        ax.set_ylim(bottom=0)

    ax = axes[n_folds]
    avg_values, std_values = [], []
    for m in metric_names:
        vals = [all_fold_metrics[f][m] for f in range(n_folds)]
        avg_values.append(np.mean(vals))
        std_values.append(np.std(vals))
    ax.bar(range(len(metric_names)), avg_values, yerr=std_values,
           color=colors[:len(metric_names)], capsize=4)
    ax.set_title("Avg. Across Folds", fontsize=13)
    ax.set_xticks(range(len(metric_names)))
    ax.set_xticklabels(metric_names, rotation=45, ha="right", fontsize=9)
    ax.set_ylim(bottom=0)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "evaluation_barplots.png"), dpi=150)
    plt.show()

## Kaggle Submission Helper

In [ ]:
def save_predictions_csv(pred_vectors, filepath):
    """
    Save predictions in Kaggle submission format.
    pred_vectors: (N, 35778) array -> flatten to 1D, ID from 1..N*35778
    """
    flat = pred_vectors.flatten()
    ids = np.arange(1, len(flat) + 1)
    df = pd.DataFrame({"ID": ids, "Predicted": flat})
    df.to_csv(filepath, index=False)
    print(f"Saved predictions to {filepath} ({len(flat)} entries)")

## Load Data

In [ ]:
data_dir = "dgl-2026-brain-graph-super-resolution-challenge"
lr_path = os.path.join(data_dir, "lr_train.csv")
hr_path = os.path.join(data_dir, "hr_train.csv")
test_lr_path = os.path.join(data_dir, "lr_test.csv")

print("Loading data...")
lr_matrices, hr_vecs = load_data(lr_path, hr_path)
print(f"  LR matrices: {lr_matrices.shape}, HR vectors: {hr_vecs.shape}")

## Hyperparameters

In [ ]:
hidden_dims = [256, 256, 256]
edge_dim = 128
dropout = 0.1
epochs = 500
learning_rate = 0.001
output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

## 3-Fold Cross-Validation

In [ ]:
start_time = time.time()

kf = KFold(n_splits=3, shuffle=True, random_state=random_seed)

all_fold_metrics = []
fold_models = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(lr_matrices)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1}/3")
    print(f"{'='*60}")
    print(f"  Train: {len(train_idx)} samples, Val: {len(val_idx)} samples")

    train_lr = lr_matrices[train_idx]
    train_hr = hr_vecs[train_idx]
    val_lr = lr_matrices[val_idx]
    val_hr = hr_vecs[val_idx]

    model = BrainGCN_SR(
        lr_dim=160, hr_dim=268,
        hidden_dims=hidden_dims,
        edge_dim=edge_dim,
        dropout=dropout,
    )
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Model parameters: {total_params:,}")

    train_model(model, train_lr, train_hr, val_lr, val_hr,
                epochs=epochs, lr=learning_rate, device=device.type)

    fold_models.append(model)

    pred_vecs = predict(model, val_lr, device=device.type)

    csv_path = os.path.join(output_dir, f"predictions_fold_{fold_idx + 1}.csv")
    save_predictions_csv(pred_vecs, csv_path)

    print(f"\n  Evaluating fold {fold_idx + 1}...")
    metrics = compute_evaluation_metrics(pred_vecs, val_hr)
    all_fold_metrics.append(metrics)

    print(f"  Fold {fold_idx + 1} results:")
    for name, val in metrics.items():
        print(f"    {name}: {val:.6f}")

total_time = time.time() - start_time
print(f"\nTotal 3F-CV training time: {total_time:.1f}s")

## Results Summary & Plots

In [ ]:
print(f"{'='*60}")
print("AVERAGE ACROSS FOLDS")
print(f"{'='*60}")
metric_names = list(all_fold_metrics[0].keys())
for m in metric_names:
    vals = [all_fold_metrics[f][m] for f in range(3)]
    print(f"  {m}: {np.mean(vals):.6f} +/- {np.std(vals):.6f}")

plot_fold_results(all_fold_metrics, output_dir=output_dir)

## Generate Kaggle Submission (Ensemble of Fold Models)

In [ ]:
print(f"{'='*60}")
print(f"GENERATING KAGGLE TEST PREDICTIONS (ensemble of {len(fold_models)} fold models)")
print(f"{'='*60}")

test_lr_matrices = load_test_data(test_lr_path)
print(f"  Test LR matrices: {test_lr_matrices.shape}")

ensemble_preds = []
for i, fold_model in enumerate(fold_models):
    print(f"  Predicting with fold {i+1} model...")
    preds = predict(fold_model, test_lr_matrices, device=device.type)
    ensemble_preds.append(preds)

test_pred_vecs = np.mean(ensemble_preds, axis=0)
print(f"  Ensemble averaging complete ({len(fold_models)} models)")

kaggle_path = os.path.join(output_dir, "kaggle_submission.csv")
save_predictions_csv(test_pred_vecs, kaggle_path)